# 💿 Dataset Conversion (Sensor-Timestamp Synchronized)

This notebook converts raw robot recordings (`.mcap` files) into the LeRobot format using **sensor-timestamp synchronization**.

## Key Differences from Standard Conversion

| Feature | Standard (`01_create_dataset.ipynb`) | Synced (this notebook) |
|---------|--------------------------------------|------------------------|
| Sync method | Log-time (arrival order) | Sensor timestamps |
| Frame generation | Based on message arrival | Synthetic timestamps at fixed FPS |
| Slow topics | May cause gaps | Reuses messages within tolerance |

**When to use this notebook:**
- Your sensors have varying frequencies
- You need precise temporal alignment across topics
- Standard conversion produces frame misalignment

The process involves:
1. **Exploring** the available raw data.
2. **Configuring** the dataset parameters (including sync tolerance).
3. **Running** the synchronized conversion.

--- 
## 1. Explore Raw Data

First, let's list the available raw data directories. Each directory contains a set of `.mcap` files from different teleoperation sessions.

In [ ]:
!du -sh ../data/raw/*

--- 
## 2. Configure Conversion

Now, specify the input and output paths and define the dataset's structure.

### Sync-Specific Parameters

- **`TOLERANCE_MS`**: Maximum time difference (in milliseconds) allowed when matching messages. If `None`, it is auto-computed as `1000 / target_fps` (i.e., one frame period).

> **Action Required:** Update `RAW_DATA_DIR`, `OUTPUT_DIR`, and optionally `TOLERANCE_MS` below.

In [ ]:
import pathlib
from example_policies.data_ops.config.pipeline_config import PipelineConfig
from example_policies.utils.action_order import ActionMode

# --- Paths ---
# TODO: Set the input directory containing your .mcap files.
RAW_DATA_DIR = pathlib.Path("../data/raw/raw_data_test")

# TODO: Set your desired output directory name.
OUTPUT_DIR = pathlib.Path("../data/lerobot/output_test_synced")

# --- Sync-Specific Configuration ---
# Maximum time difference for matching messages across topics.
# Set to None for auto-compute (1 frame period at target_fps).
TOLERANCE_MS = None  # e.g., 40.0 for 40ms tolerance

# Maximum number of episodes to process (None = no limit)
MAX_EPISODES = None  # e.g., 5 to process only first 5 episodes

# --- Configuration ---
# TODO: A descriptive label for the task, used for VLA-style text conditioning.
TASK_LABEL = "pick and stack the [TODO] colored lego block on the white lego block"

cfg = PipelineConfig(
    task_name=TASK_LABEL,
    # Observation features to include in the dataset.
    include_tcp_poses=True,
    include_rgb_images=True,
    include_depth_images=False,
    include_last_command=True,
    # Action representation. TCP is a good default.
    action_level=ActionMode.DELTA_TCP,
    # Subsampling and filtering. These are task-dependent.
    recording_fps=20,
    target_fps=10,
    max_pause_seconds=0.2,
    min_episode_seconds=1,
)

# Compute actual tolerance
actual_tolerance_ms = TOLERANCE_MS if TOLERANCE_MS is not None else (1000.0 / cfg.target_fps)

print(f"Input path:  {RAW_DATA_DIR}")
print(f"Output path: {OUTPUT_DIR}")
print(f"\nSync Configuration:")
print(f"  - Target FPS: {cfg.target_fps} Hz")
print(f"  - Tolerance: {actual_tolerance_ms:.1f} ms {'(auto)' if TOLERANCE_MS is None else ''}")
print(f"  - Max episodes: {MAX_EPISODES if MAX_EPISODES else 'All'}")

--- 
## 2.5 Debug: Analyze Topic Frequencies

Before running the conversion, let's analyze the raw MCAP files to see what topics are available and at what frequencies they're recorded. This is especially important for synced conversion to ensure your tolerance setting is appropriate.

In [ ]:
from mcap.reader import make_reader
from collections import defaultdict
import numpy as np
import sys
from example_policies.data_ops.config.rosbag_topics import RosTopicEnum
from example_policies.data_ops.utils.conversion_utils import get_selected_episodes

def analyze_mcap_topics(mcap_path):
    """Analyze topics and their frequencies in an MCAP file."""
    topic_stats = defaultdict(lambda: {"count": 0, "timestamps": []})
    
    with open(mcap_path, 'rb') as f:
        reader = make_reader(f)
        for schema, channel, message in reader.iter_messages():
            topic = channel.topic
            timestamp = message.log_time  # nanoseconds
            topic_stats[topic]["count"] += 1
            topic_stats[topic]["timestamps"].append(timestamp)
    
    # Calculate frequencies
    results = {}
    for topic, stats in topic_stats.items():
        if len(stats["timestamps"]) > 1:
            timestamps_sec = np.array(stats["timestamps"]) / 1e9  # convert to seconds
            time_diffs = np.diff(timestamps_sec)
            
            # Count invalid time diffs
            invalid_count = np.sum(time_diffs <= 0)
            
            # Filter out zero or negative time diffs
            valid_diffs = time_diffs[time_diffs > 0]
            
            if len(valid_diffs) > 0:
                avg_freq = 1.0 / np.mean(valid_diffs)
                min_freq = 1.0 / np.max(valid_diffs)  # Longest gap = lowest frequency
                max_freq = 1.0 / np.min(valid_diffs)  # Shortest gap = highest frequency
                std_freq = np.std(1.0 / valid_diffs)  # Std of instantaneous frequencies
            else:
                avg_freq = 0
                min_freq = 0
                max_freq = 0
                std_freq = 0
            
            results[topic] = {
                "message_count": stats["count"],
                "avg_frequency_hz": avg_freq,
                "min_frequency_hz": min_freq,
                "max_frequency_hz": max_freq,
                "std_frequency_hz": std_freq,
                "duration_sec": timestamps_sec[-1] - timestamps_sec[0],
                "invalid_time_diffs": int(invalid_count)
            }
        else:
            results[topic] = {
                "message_count": stats["count"],
                "avg_frequency_hz": 0,
                "min_frequency_hz": 0,
                "max_frequency_hz": 0,
                "std_frequency_hz": 0,
                "duration_sec": 0,
                "invalid_time_diffs": 0
            }
    
    return results

# Define topics of interest with new and old names
TOPICS_OF_INTEREST = [
    ("RGB Left Image", [RosTopicEnum.RGB_LEFT_IMAGE.value]),
    ("RGB Right Image", [RosTopicEnum.RGB_RIGHT_IMAGE.value]),
    ("RGB Static Image", [RosTopicEnum.RGB_STATIC_IMAGE.value]),
    ("Desired Gripper Left", [RosTopicEnum.DES_GRIPPER_LEFT.value]),
    ("Desired Gripper Right", [RosTopicEnum.DES_GRIPPER_RIGHT.value]),
    ("Desired TCP Left", ["/cartesian_target_left", "/desired_pose_twist_left"]),
    ("Desired TCP Right", ["/cartesian_target_right", "/desired_pose_twist_right"]),
    ("Actual Joint State", [RosTopicEnum.ACTUAL_JOINT_STATE.value]),
    ("Actual TCP Left", [RosTopicEnum.ACTUAL_TCP_LEFT.value]),
    ("Actual TCP Right", [RosTopicEnum.ACTUAL_TCP_RIGHT.value]),
]

# Get valid episodes using the standard function
episode_paths = get_selected_episodes(RAW_DATA_DIR, success_only=True)

# TODO: Change this index to analyze a different episode
SELECTED_EPISODE_INDEX = 0

# Build output as a single string to avoid duplicate printing
output = []
output.append(f"Found {len(episode_paths)} valid episodes\n")

if episode_paths:
    # Validate index
    if SELECTED_EPISODE_INDEX >= len(episode_paths):
        SELECTED_EPISODE_INDEX = 0
        output.append(f"⚠️  Index out of range, using episode 0\n")
    
    selected_episode_path = episode_paths[SELECTED_EPISODE_INDEX]
    output.append(f"Analyzing MCAP file [{SELECTED_EPISODE_INDEX}]: {selected_episode_path.name}\n")
    output.append("="*80 + "\n")
    
    topic_stats = analyze_mcap_topics(selected_episode_path)
    
    # Check if tolerance is appropriate
    output.append(f"\n🔧 Sync Tolerance Check (configured: {actual_tolerance_ms:.1f} ms):\n")
    output.append("-"*80 + "\n")
    
    for topic_label, topic_names in TOPICS_OF_INTEREST:
        found_topic = None
        for topic_name in topic_names:
            if topic_name in topic_stats:
                found_topic = topic_name
                break
        
        if found_topic:
            stats = topic_stats[found_topic]
            avg_period_ms = 1000.0 / stats['avg_frequency_hz'] if stats['avg_frequency_hz'] > 0 else float('inf')
            
            # Check if tolerance is appropriate for this topic
            if avg_period_ms > actual_tolerance_ms * 2:
                status = "⚠️  SLOW (may miss frames)"
            elif avg_period_ms > actual_tolerance_ms:
                status = "⚡ OK (some reuse)"
            else:
                status = "✅ FAST"
            
            output.append(f"  {topic_label}: {stats['avg_frequency_hz']:.1f} Hz (period: {avg_period_ms:.1f} ms) - {status}\n")
    
    output.append("\n" + "="*80 + "\n")
    
    # Print all topics found
    output.append("\n📋 All topics in MCAP file:\n")
    output.append("-"*80 + "\n")
    for topic, stats in sorted(topic_stats.items()):
        output.append(f"  {topic}: {stats['message_count']} msgs, {stats['avg_frequency_hz']:.2f} Hz\n")
else:
    output.append(f"No valid episodes found in {RAW_DATA_DIR}\n")
    output.append(f"💡 Try running with success_only=False to see all files\n")

# Print all at once
sys.stdout.write(''.join(output))
sys.stdout.flush()

--- 
## 2.6 Visualize Topic Time Step Distribution

Visualize the distribution of time steps (time differences between consecutive messages) for a specific topic. This helps determine if your tolerance setting is appropriate.

In [ ]:
import matplotlib.pyplot as plt
from mcap.reader import make_reader
from collections import defaultdict

# Configuration
FIX_X_AXIS_RANGE = False  # Set to False to show full data range
X_AXIS_RANGE_PERCENT = 50  # Show mean ± this percentage

# Create plots for all topics of interest
if episode_paths:
    # Use the same episode selected in the previous cell
    print(f"Visualizing episode [{SELECTED_EPISODE_INDEX}]: {selected_episode_path.name}")
    print(f"Configured tolerance: {actual_tolerance_ms:.1f} ms (shown as vertical red line)\n")
    
    # Iterate through all topics of interest
    for topic_label, topic_names in TOPICS_OF_INTEREST:
        # Find which version of the topic exists and collect timestamps
        found_topic = None
        timestamps = []
        
        for topic_name in topic_names:
            timestamps = []
            with open(selected_episode_path, 'rb') as f:
                reader = make_reader(f)
                for schema, channel, message in reader.iter_messages():
                    if channel.topic == topic_name:
                        timestamps.append(message.log_time / 1e9)  # Convert to seconds
            
            if len(timestamps) > 0:
                found_topic = topic_name
                break
        
        if not found_topic or len(timestamps) <= 1:
            continue
        
        # Calculate time differences (in milliseconds for better readability)
        time_diffs_ms = np.diff(timestamps) * 1000
        
        # Filter out invalid diffs for visualization
        valid_diffs = time_diffs_ms[time_diffs_ms > 0]
        
        if len(valid_diffs) > 0:
            # Create the distribution plot
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
            
            # Histogram
            ax1.hist(valid_diffs, bins=50, edgecolor='black', alpha=0.7)
            ax1.axvline(x=actual_tolerance_ms, color='red', linestyle='--', linewidth=2, label=f'Tolerance ({actual_tolerance_ms:.1f} ms)')
            ax1.set_xlabel('Time Step (ms)', fontsize=12)
            ax1.set_ylabel('Count', fontsize=12)
            ax1.set_title(f'Distribution of Time Steps\n{topic_label} ({found_topic})', fontsize=14)
            ax1.grid(True, alpha=0.3)
            ax1.legend()
            
            # Optionally fix x-axis range to mean ± percentage
            mean_val = np.mean(valid_diffs)
            if FIX_X_AXIS_RANGE:
                x_min = mean_val * (1 - X_AXIS_RANGE_PERCENT / 100)
                x_max = mean_val * (1 + X_AXIS_RANGE_PERCENT / 100)
                ax1.set_xlim(x_min, x_max)
            
            # Add statistics text
            stats_text = f'Mean: {np.mean(valid_diffs):.2f} ms\n'
            stats_text += f'Median: {np.median(valid_diffs):.2f} ms\n'
            stats_text += f'Std: {np.std(valid_diffs):.2f} ms\n'
            stats_text += f'Min: {np.min(valid_diffs):.2f} ms\n'
            stats_text += f'Max: {np.max(valid_diffs):.2f} ms\n'
            stats_text += f'Freq: {1000/np.mean(valid_diffs):.2f} Hz'
            ax1.text(0.98, 0.98, stats_text, transform=ax1.transAxes,
                    verticalalignment='top', horizontalalignment='right',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
                    fontsize=10)
            
            # Box plot for outlier detection (horizontal)
            bp = ax2.boxplot(valid_diffs, vert=False)
            ax2.axvline(x=actual_tolerance_ms, color='red', linestyle='--', linewidth=2, label=f'Tolerance ({actual_tolerance_ms:.1f} ms)')
            ax2.set_xlabel('Time Step (ms)', fontsize=12)
            ax2.set_title('Box Plot (Outlier Detection)', fontsize=14)
            ax2.grid(True, alpha=0.3, axis='x')
            ax2.legend()
            
            # Match x-axis range with histogram if fixed
            if FIX_X_AXIS_RANGE:
                ax2.set_xlim(x_min, x_max)
            
            plt.tight_layout()
            plt.show()
            
            print(f"Analyzed {len(timestamps)} messages for {topic_label}")
            print(f"Valid time steps: {len(valid_diffs)}, Invalid: {len(time_diffs_ms) - len(valid_diffs)}\n")
else:
    print(f"No MCAP files found in {RAW_DATA_DIR}")

--- 
## 3. Run Synchronized Conversion

This cell executes the **sensor-timestamp synchronized** conversion process. 

### How it works:
1. **Ingest all messages** from the MCAP file with their sensor timestamps
2. **Generate synthetic timestamps** at a fixed output frequency (`target_fps`)
3. **Find nearest messages** from all topics within the tolerance window
4. **Assemble frames** and write to LeRobot format

This may take a while depending on the size of your data.

In [ ]:
from example_policies.data_ops.dataset_conversion_synced import (
    convert_episodes_synced,
    print_conversion_result,
)
from example_policies.data_ops.utils.conversion_utils import get_selected_episodes

# Get episodes
episode_paths = get_selected_episodes(RAW_DATA_DIR, success_only=True)

# Optionally limit number of episodes
if MAX_EPISODES is not None:
    episode_paths = episode_paths[:MAX_EPISODES]
    print(f"Limiting to first {MAX_EPISODES} episodes")

print(f"Converting {len(episode_paths)} episodes...")
print(f"Sync settings: target_fps={cfg.target_fps}, tolerance={actual_tolerance_ms:.1f}ms\n")

# Run synchronized conversion
result = convert_episodes_synced(
    episode_paths=episode_paths,
    output_dir=OUTPUT_DIR,
    config=cfg,
    tolerance_ms=actual_tolerance_ms,
)

# Print results
print_conversion_result(result)

---
## 4. Verify Output

Let's verify the converted dataset was created successfully.

In [ ]:
import json

# Check output directory contents
print(f"Output directory: {OUTPUT_DIR}")
print("\nContents:")
!ls -la {OUTPUT_DIR}

# Show metadata if available
metadata_path = OUTPUT_DIR / "meta" / "info.json"
if metadata_path.exists():
    print("\n" + "="*50)
    print("Dataset Info:")
    print("="*50)
    with open(metadata_path) as f:
        info = json.load(f)
    print(f"  FPS: {info.get('fps', 'N/A')}")
    print(f"  Total episodes: {info.get('total_episodes', 'N/A')}")
    print(f"  Total frames: {info.get('total_frames', 'N/A')}")
    print(f"  Features: {list(info.get('features', {}).keys())}")

--- 
## ✅ Done!

Your new **synchronized** dataset is ready at the output path you specified. 

**Next steps:**
- Proceed to `02_train_model_simple.ipynb` to train a policy
- Use `04_upload_and_visualize_datasets.ipynb` to visualize the dataset

**Troubleshooting:**
- If you see many "skipped pauses", the robot may have been idle for extended periods
- If frames are missing, try increasing `TOLERANCE_MS`
- If frames are duplicated/stuttering, try decreasing `TOLERANCE_MS`